In [1]:
import glob
import pickle

from tqdm import tqdm


In [2]:

paths = {
    'k_16': "../runs/minif2f/fixed_no_score-k16/2024_08_28/16_02_42/traces/0/",
    'bestfs': "../runs/minif2f/minif2f_bestfs/2024_07_31/16_28_47/traces/0/",
}


In [3]:
def load_traces(path):
    files = glob.glob(path + '*', recursive=True)
    traces_ = []

    for file in tqdm(files):
        try:
            traces_.append(pickle.load(open(file, "rb")))
        except:
            # print ('Failed to load:', file)
            continue

    return traces_


traces = {}

for key, path in paths.items():
    traces[key] = load_traces(path)



100%|██████████| 244/244 [00:19<00:00, 12.24it/s]


In [4]:
# name for theorem for demo
demo_thm = 'algebra_manipexpr_apbeq2cceqiacpbceqm2'

In [5]:
from experiments.end_to_end.proof_node import ErrorNode

dpp_attempt = [t for t in traces['k_16'] if t.theorem.full_name == demo_thm][0]
bestfs_attempt = [t for t in traces['bestfs'] if t.theorem.full_name == demo_thm][0]

print(dpp_attempt)
print(bestfs_attempt)

# dpp_attempt
# bestfs_attempt
bestfs_res = [(t.tactic, t.dst[0].goal) for t in bestfs_attempt.tree.out_edges if not isinstance(t.dst[0], ErrorNode)]
for x in bestfs_res:
    print(x)
print('\n\n')
dpp_res = [(t.tactic, t.dst[0].goal) for t in dpp_attempt.tree.out_edges if not isinstance(t.dst[0], ErrorNode)]
for x in dpp_res:
    print(x)

print([t.tactic for t in bestfs_attempt.tree.out_edges])
print([t.tactic for t in dpp_attempt.tree.out_edges])

[(i, x) for i, x in enumerate(dpp_attempt.tree.data['original_tacs'])]

# number of unique, number successful for each
print(len(set([x[1] for x in dpp_res])), len(set([x[1] for x in bestfs_res])), len(bestfs_res), len(dpp_res))

SearchResult(theorem=Theorem(repo=LeanGitRepo(url='https://github.com/facebookresearch/miniF2F', commit='5271ddec788677c815cf818a06f368ef6498a106'), file_path=PosixPath('lean/src/valid.lean'), full_name='algebra_manipexpr_apbeq2cceqiacpbceqm2'), status=<Status.PROVED: 'Proved'>, proof=['subst c', 'ring', 'ring', 'simp [h₀]', 'ring', 'norm_num'], tree=InternalNode(goal='a b c : ℂ,\nh₀ : a + b = 2 * c,\nh₁ : c = complex.I\n⊢ a * c + b * c = -2', _status=<Status.PROVED: 'Proved'>), total_time=1643.7819045372307, tac_time=1052.9874012456276, search_time=0.23241461627185345, env_time=590.5511524891481, num_expansions=2416, num_nodes=821)
SearchResult(theorem=Theorem(repo=LeanGitRepo(url='https://github.com/facebookresearch/miniF2F', commit='5271ddec788677c815cf818a06f368ef6498a106'), file_path=PosixPath('lean/src/valid.lean'), full_name='algebra_manipexpr_apbeq2cceqiacpbceqm2'), status=<Status.OPEN: 'Open'>, proof=None, tree=InternalNode(goal='a b c : ℂ,\nh₀ : a + b = 2 * c,\nh₁ : c = compl

In [13]:
# [(i, t) for i, t in enumerate(dpp_attempt.tree.data['original_tacs'])]

[(0, ('rw [h₁, h₂]', -3.445507049560547)),
 (1, ('linarith', -3.7793476581573486)),
 (2, ('rw [h₁, h₀]', -3.834747314453125)),
 (3, ('simp [h₁, h₀]', -4.279934883117676)),
 (4, ('ring', -4.394063949584961)),
 (5, ('simp [h₁, h₂]', -4.495162010192871)),
 (6, ('simp [h₀]', -4.759472370147705)),
 (7, ('rw h₁', -4.9020795822143555)),
 (8, ('linear_combination h₀', -5.03296422958374)),
 (9, ('exact h₀', -5.036291599273682)),
 (10, ('rw [h₁, h₀, <a>mul_add</a>]', -5.259109973907471)),
 (11, ('rw [h₁, h₀, <a>two_mul</a>]', -5.487335681915283)),
 (12, ('exfalso', -5.493969917297363)),
 (13, ('rw [h₁, h₁]', -5.523379802703857)),
 (14, ('rw [h₁, h₀, <a>add_mul</a>]', -5.656346797943115)),
 (15, ('simp [h₁]', -5.688459396362305)),
 (16, ('subst h₁', -5.766551494598389)),
 (17, ('rw [h₁, h₀, <a>mul_comm</a>]', -5.858577728271484)),
 (18, ('rw [h₁, h₂] at h₀', -5.864826202392578)),
 (19, ('rw [h₁, h₀, <a>mul_assoc</a>]', -5.91461706161499)),
 (20, ('rw h₁ at h₀', -5.987144470214844)),
 (21, ('rwa h

In [6]:
# 24 (subst c), 39 (symmetry), 7 (rw h1), 15 (simp [h1]), 0 (rw [h1, h2]) 
ids = [24, 39, 7, 15]  #, 0]#, 5]

In [7]:
dpp_attempt.tree.data['similarity_scores'][ids][:, ids]

array([[0.99998224, 0.66776556, 0.63511086, 0.58976644],
       [0.66776556, 1.000002  , 0.5602119 , 0.48083457],
       [0.63511086, 0.5602119 , 1.0000266 , 0.74575335],
       [0.58976644, 0.48083457, 0.74575335, 0.99999255]], dtype=float32)

In [18]:

from models.end_to_end.tactic_models.diversity_model.model import DiversityModel

config = {
    'ckpt_dir': '../runs/error_pred/combined_transition_model/combined_minif2f_valid.ckpt',
    'model': 'sean-lamont/leandojo-lean3-reprover-novel-premises',
    'max_seq_len': 2000,
    'num_filtered': 2,
    'temperature': 1,
    'p': 0.75,
    'score_network': False,
    'error_weight': 0.5,
    'time_weight': 0.1,
    'error_only': False,
    'fixed_size': True}

# convert config to omegaconf

from omegaconf import OmegaConf

config = OmegaConf.create(config)

model = DiversityModel(config, 'cuda')



Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [19]:
import torch

tactics = dpp_attempt.tree.data['original_tacs']
temperature = 1
scale = 1
theorem = dpp_attempt.theorem.full_name
state = dpp_attempt.tree.data['augmented_state']
with torch.no_grad():
    encs = []

    logprobs = [t[1] / temperature for t in tactics]

    # get softmax over logprobs
    probs = torch.softmax(torch.tensor(logprobs), dim=0) * scale
    probs = probs.to('cuda')

    # chunking gives slight speedup, but high memory cost
    chunk_size = 1
    for ind in range(0, len(tactics), chunk_size):
        t = [t[0] for t in tactics[ind:ind + chunk_size]]

        goals = [t_ + theorem + '\n\n' + state for t_ in t]

        tokenized_goals = model.tokenizer(
            goals,
            padding="longest",
            max_length=int(model.max_seq_len * 1.5),
            truncation=True,
            return_tensors="pt", )

        tokenized_tactics = model.tokenizer(
            t,
            padding="longest",
            max_length=model.max_seq_len,
            truncation=True,
            return_tensors="pt",
        )

        lens = tokenized_tactics.attention_mask.sum(dim=1)

        if not model.autoencoder:
            enc = model.get_tac_encoding(tokenized_goals.input_ids.to(model.device),
                                         tokenized_goals.attention_mask.to(model.device), lens.to(model.device))
        else:
            enc = model.get_autoencoder_encoding(tokenized_tactics.input_ids.to(model.device),
                                                 tokenized_tactics.attention_mask.to(model.device))

        encs.append(enc)

    vec_matrix = torch.cat(encs, dim=0)
vec_matrix = torch.mul(vec_matrix, probs.unsqueeze(1)).cpu().numpy()


In [75]:
import numpy as np

test_encs = np.array([encs[i][0].cpu().numpy() for i in ids])

from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_2d = pca.fit_transform(test_encs)




In [165]:
probs[ids]


tensor([0.0095, 0.0026, 0.0348, 0.0158], device='cuda:0')

In [164]:
import plotly.graph_objects as go

# scale vectors by probs[ids]
vectors = X_2d * probs.cpu().numpy()[ids][:, None]

# Create origin points
origin_x = np.zeros(vectors.shape[0])
origin_y = np.zeros(vectors.shape[0])

# Create a figure for Plotly
fig = go.Figure()

# custom position dict for text labels
# text_pos = {1: 'top left', 2: 'top right', 3: 'bottom right', 4: 'top center'}

text_pos = {1: 'top left', 2: 'top right', 3: 'bottom right', 4: 'top left'}

# 3 == simp, 0 = subst, 1 = symmetry, 2 = rw

import plotly.io as pio

# Add vectors as arrows
for i in range(vectors.shape[0]):
    fig.add_trace(go.Scatter(
        x=[origin_x[i], vectors[i, 0] * 0.9],
        y=[origin_y[i], vectors[i, 1] * 0.9],
        mode="lines",
        marker=dict(size=5, color='black'),
        line=dict(width=2, color='black'),
        showlegend=False
    ))

    for j in range(i + 1, vectors.shape[0]):
        if i == 1 and j == 3:
            pass
        else:
            fig.add_trace(go.Scatter(
                x=[vectors[i, 0], vectors[j, 0]],
                y=[vectors[i, 1], vectors[j, 1]],
                mode="lines",
                line=dict(width=1, color='black', dash='dash'),  # Dashed line style
                showlegend=False
            ))

            if i == 0 and j == 3:
                # Add shaded region
                # Create a shaded region between dashed lines
                fig.add_trace(go.Scatter(
                    x=[vectors[i, 0], vectors[j, 0], vectors[2, 0], vectors[i, 0]],
                    y=[vectors[i, 1], vectors[j, 1], vectors[2, 1], vectors[i, 1]],
                    fill='toself',
                    fillcolor='rgba(100, 0, 80, 0.3)',  # Shaded color
                    line=dict(color='rgba(0, 100, 80, 0)'),  # No border line for fill
                    mode='lines',
                    showlegend=False
                ))

            if i == 1 or j == 1:
                # Add shaded region
                # Create a shaded region between dashed lines
                fig.add_trace(go.Scatter(
                    x=[vectors[i, 0], vectors[j, 0], origin_x[i], vectors[i, 0]],
                    y=[vectors[i, 1], vectors[j, 1], origin_y[i], vectors[i, 1]],
                    fill='toself',
                    fillcolor='rgba(0, 100, 80, 0.3)',  # Shaded color
                    line=dict(color='rgba(0, 100, 80, 0)'),  # No border line for fill
                    mode='lines',
                    showlegend=False
                ))

            # Create a shaded region between dashed lines
            fig.add_trace(go.Scatter(
                x=[vectors[i, 0], vectors[j, 0], origin_x[i], vectors[i, 0]],
                y=[vectors[i, 1], vectors[j, 1], origin_y[i], vectors[i, 1]],
                fill='toself',
                fillcolor='rgba(100, 100, 80, 0.1)',  # Shaded color
                line=dict(color='rgba(0, 100, 80, 0)'),  # No border line for fill
                mode='lines',
                showlegend=False
            ))

    # Add arrowheads using annotations
    fig.add_annotation(
        ax=origin_x[i], ay=origin_y[i],
        axref="x", ayref="y",
        x=vectors[i, 0], y=vectors[i, 1],
        xref="x", yref="y",
        showarrow=True,
        arrowhead=3,  # Style of the arrowhead
        arrowsize=2,  # Size of the arrowhead
        arrowwidth=2,  # Thickness of the arrow shaft
        arrowcolor="black"
    )
    
# Set the layout to make it more aesthetic
fig.update_layout(
    xaxis=dict(
        range=[min(vectors[:, 0]), max(vectors[:, 0]) * 2],
        showgrid=False,  # Hide the grid lines
        # gridcolor='rgba(0, 0, 0, 0.2)',  # Transparent grid lines color
        showline=False,  # Hide the axis line
        # showgrid=False,  # Hide the grid lines
        zeroline=False,  # Hide the zero line
        showticklabels=False,  # Hide the tick labels
    ),
    yaxis=dict(
        # range=[min(vectors[:, 1])*1.5, max(vectors[:,1])*3],
        showgrid=False,  # Hide the grid lines
        # gridcolor='rgba(0, 0, 0, 0.2)',  # Transparent grid lines color
        showline=False,  # Hide the axis line
        # showgrid=False,  # Hide the grid lines
        zeroline=False,  # Hide the zero line
        showticklabels=False,  # Hide the tick labels
    ),
    width=600,
    height=600,
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="white",
)

# fig.show()
fig.write_image("vector_plot_scaled.svg")  #, format="pdf")

In [163]:

# scale X_2d to have unit norm
vectors = X_2d / np.linalg.norm(X_2d, axis=1)[:, None]


# Create origin points
origin_x = np.zeros(vectors.shape[0])
origin_y = np.zeros(vectors.shape[0])

# Create a figure for Plotly
fig = go.Figure()

# custom position dict for text labels
text_pos = {1: 'top left', 2: 'top right', 3: 'bottom right', 4: 'top left'}

# 3 == simp, 0 = subst, 1 = symmetry, 2 = rw

# Add vectors as arrows
for i in range(vectors.shape[0]):
    fig.add_trace(go.Scatter(
        x=[origin_x[i], vectors[i, 0] * 0.9],
        y=[origin_y[i], vectors[i, 1] * 0.9],
        mode="lines",
        marker=dict(size=5, color='black'),
        line=dict(width=2, color='black'),
        showlegend=False
    ))

    # Add arrowheads using annotations
    fig.add_annotation(
        ax=origin_x[i], ay=origin_y[i],
        axref="x", ayref="y",
        x=vectors[i, 0], y=vectors[i, 1],
        xref="x", yref="y",
        showarrow=True,
        arrowhead=3,  # Style of the arrowhead
        arrowsize=2,  # Size of the arrowhead
        arrowwidth=2,  # Thickness of the arrow shaft
        arrowcolor="black"
    )

# Set the layout to make it more aesthetic
fig.update_layout(
    xaxis=dict(
        showline=False,  # Hide the axis line
        showgrid=False,  # Hide the grid lines
        # gridcolor='rgba(0, 0, 0, 0.2)',  # Transparent grid lines color
        zeroline=False,  # Hide the zero line
        showticklabels=False,  # Hide the tick labels
    ),
    yaxis=dict(
        # range=[min(vectors[:, 1])*1.5, max(vectors[:,1])*3],
        showline=False,  # Hide the axis line
        showgrid=False,  # Hide the grid lines
        # gridcolor='rgba(0, 0, 0, 0.2)',  # Transparent grid lines color
        zeroline=False,  # Hide the zero line
        showticklabels=False,  # Hide the tick labels
    ),
    width=600,
    height=600,
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="white",
)

# fig.show()
fig.write_image("vector_plot.svg")  #, format="pdf")